In [1]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime

# Import your modules
from data_collection import CAPMProcessor
from portfolio_tracker import PortfolioTracker

# Import Standard Algorithms
from Algo_1 import VarianceSelector
from Algo_2 import FundamentalScoreSelector
from Algo_4 import TechnicalScoreSelector
from Algo_xg import SmartPortfolioBuilder
from MVPTs import MVPTOptimizer

# Import Naive Algorithms
from Naive_Algo import (
    NaiveVarianceSelector,
    NaiveFundamentalSelector,
    NaiveTechnicalScoreSelector,
    NaiveSmartPortfolioBuilder
)

In [2]:
# --- Configuration ---
TICKERS = [
    'AAL', 'AAPL', 'ALNY', 'AMD', 'AMZN', 'AVPT', 'BA', 'BBIO', 'BLD', 'BYND', 
    'C', 'CFLT',  'CHWY', 'CLF', 'COST', 'DASH', 'DDD', 'DG', 'DIS', 'DKNG', 'EBS', 'EMXC', 'ESTC', 'FIS',
    'FLUT', 'GE', 'GOOGL',  'HD', 'INTC', 'ISRG', 'ITRG', 'JNJ', 'JPM', 'KHC', 'KO', 'KR', 'LGMK', 'LMT', 
    'LOCO', 'LSCC', 'LUNR', 'MDB', 'META',  'MSFT', 'MSTR', 'NFLX', 'NKE', 'NVAX', 'NVDA', 'PEP', 'PFE', 'PGR',
    'PLUG', 'PTON', 'PYPL', 'QCOM', 'ROKU',  'SBUX', 'SCHR', 'SPG', 'SPOT', 'SPWR', 'SWKS', 
    'T', 'TGT', 'TPR', 'TSLA', 'ULTA', 'V', 'VZ', 'WFC',  'WMT', 'XOM', 'ZM', 'DIS', 'ABNB','CRWD', 'WBD', 'TMUS'
]
MARKETS = ['^IXIC', '^NYA', '^GSPC']

# Training Period (Data Collection)
START_DATE = "2018-01-01"
END_DATE = "2021-01-01" 

# Trading Parameters
CAPITAL = 5000
RISK_FREE_RATE = 0.036  

# *** NEW: Explicit Sell Date for Backtesting ***
SELL_DATE = "2021-09-01"

In [3]:
# Create output directory
OUTPUT_DIR = "AiDAS portfolio_results (2024-06-01)"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [4]:
def save_result(df, filename_prefix, tracker, tag):
    """Helper to save allocation and performance to CSV."""
    if df.empty:
        print(f"  [Skipped] {tag} produced no allocation.")
        return None
    
    # 1. Save Allocation
    alloc_path = os.path.join(OUTPUT_DIR, f"{filename_prefix}_Allocation.csv")
    df.to_csv(alloc_path, index=False)
    
    # 2. Track Performance (Using the specific SELL_DATE)
    print(f"  Tracking performance for {tag} on {SELL_DATE}...")
    tracked_df, summary = tracker.get_performance(df, target_date=SELL_DATE)
    
    # 3. Save Tracked Result
    track_path = os.path.join(OUTPUT_DIR, f"{filename_prefix}_Performance.csv")
    tracked_df.to_csv(track_path, index=False)
    
    # Return summary for master table
    summary['Strategy'] = tag
    return summary

def main():
    print("="*60)
    print(f"STARTING MASTER PORTFOLIO GENERATION")
    print(f"Training End: {END_DATE} | Target Sell Date: {SELL_DATE}")
    print("="*60)
    
    tracker = PortfolioTracker()
    master_summary = []

    # ---------------------------------------------------------
    # 1. CORE DATA COLLECTION (Used by Algo 1, 2, MVPT)
    # ---------------------------------------------------------
    print("\n[Phase 1] Fetching Core Data (CAPM Processor)...")
    # We fetch up to the END_DATE (June 2024) for training.
    # The PortfolioTracker will independently fetch the SELL_DATE prices later.
    capm = CAPMProcessor(TICKERS, MARKETS, START_DATE, END_DATE, RISK_FREE_RATE)
    capm.fetch_market_data()
    capm.fetch_stock_data()
    capm.fetch_fundamental_data()
    df_metrics = capm.calculate_metrics()
    
    # Save Metrics for reference
    df_metrics.to_csv(os.path.join(OUTPUT_DIR, "CAPM_Metrics.csv"), index=False)

    # ---------------------------------------------------------
    # 2. RUN ALGORITHM 1 & NAIVE 1
    # ---------------------------------------------------------
    print("\n[Phase 2] Running Algo 1 (Variance Selector)...")
    
    # Standard
    algo1 = VarianceSelector(capital=CAPITAL)
    a1_alloc, _ = algo1.process(df_metrics)
    s1 = save_result(a1_alloc, "Algo1_Standard", tracker, "Algo 1 (Standard)")
    if s1: master_summary.append(s1)

    # Naive
    naive1 = NaiveVarianceSelector(capital=CAPITAL)
    n1_alloc, _ = naive1.process(df_metrics)
    sn1 = save_result(n1_alloc, "Algo1_Naive", tracker, "Algo 1 (Naive)")
    if sn1: master_summary.append(sn1)

    # ---------------------------------------------------------
    # 3. RUN ALGORITHM 2 & NAIVE 2
    # ---------------------------------------------------------
    print("\n[Phase 3] Running Algo 2 (Fundamental Selector)...")

    # Standard
    algo2 = FundamentalScoreSelector(capital=CAPITAL)
    a2_alloc, _ = algo2.process(df_metrics)
    s2 = save_result(a2_alloc, "Algo2_Standard", tracker, "Algo 2 (Standard)")
    if s2: master_summary.append(s2)

    # Naive
    naive2 = NaiveFundamentalSelector(capital=CAPITAL)
    n2_alloc, _ = naive2.process(df_metrics)
    sn2 = save_result(n2_alloc, "Algo2_Naive", tracker, "Algo 2 (Naive)")
    if sn2: master_summary.append(sn2)

    # ---------------------------------------------------------
    # 4. RUN ALGORITHM 4 & NAIVE 4
    # ---------------------------------------------------------
    print("\n[Phase 4] Running Algo 4 (Technical Selector)...")
    # Algo 4 fetches its own data structure
    algo4 = TechnicalScoreSelector(TICKERS)
    algo4.fetch_data(start_date=START_DATE, end_date=END_DATE) # Fetch once
    algo4.compute_signals()
    
    # Standard
    a4_alloc, _ = algo4.construct_portfolio(CAPITAL)
    s4 = save_result(a4_alloc, "Algo4_Standard", tracker, "Algo 4 (Standard)")
    if s4: master_summary.append(s4)

    # Naive (Inject data to skip re-fetching)
    naive4 = NaiveTechnicalScoreSelector(TICKERS)
    naive4.data_map = algo4.data_map 
    naive4.summary_df = algo4.summary_df
    
    n4_alloc, _ = naive4.construct_portfolio(CAPITAL)
    sn4 = save_result(n4_alloc, "Algo4_Naive", tracker, "Algo 4 (Naive)")
    if sn4: master_summary.append(sn4)

    # ---------------------------------------------------------
    # 5. RUN ALGO XG & NAIVE XG
    # ---------------------------------------------------------
    print("\n[Phase 5] Running Algo XG (Smart Portfolio)...")
    algo_xg = SmartPortfolioBuilder(TICKERS, MARKETS, START_DATE, END_DATE, RISK_FREE_RATE)
    # This class is optimized to fetch only daily data
    algo_xg.fetch_stock_data() 
    
    # Standard
    xg_alloc, _ = algo_xg.optimize_portfolio(CAPITAL)
    sxg = save_result(xg_alloc, "AlgoXG_Standard", tracker, "Algo XG (Standard)")
    if sxg: master_summary.append(sxg)

    # Naive (Inject data)
    naive_xg = NaiveSmartPortfolioBuilder(TICKERS, MARKETS, START_DATE, END_DATE, RISK_FREE_RATE)
    naive_xg.daily_stock_prices = algo_xg.daily_stock_prices
    
    nxg_alloc, _ = naive_xg.optimize_portfolio(CAPITAL)
    snxg = save_result(nxg_alloc, "AlgoXG_Naive", tracker, "Algo XG (Naive)")
    if snxg: master_summary.append(snxg)

    print("\n[Phase 6] Running MVPT Optimizers...")
    
    # Requires raw daily price data (Close) from the CAPMProcessor
    if capm.daily_stock_prices is not None and not capm.daily_stock_prices.empty:
        mvpt = MVPTOptimizer(capm.daily_stock_prices)
        
        # Method 1: Ledoit-Wolf Shrinkage
        # Note: This method optimizes across all available tickers in the price_df
        res_lw = mvpt.optimize_ledoit_wolf(capital=CAPITAL)
        slw = save_result(res_lw['allocation'], "MVPT_LedoitWolf", tracker, "MVPT (Ledoit-Wolf)")
        if slw: master_summary.append(slw)
        
        # Method 2: Greedy Unrestricted
        # Finds the best 25 stocks iteratively with no weight limits
        res_gru = mvpt.optimize_greedy_unrestricted(capital=CAPITAL, num_stocks=25)
        sgru = save_result(res_gru['allocation'], "MVPT_Greedy_Unrestricted", tracker, "MVPT (Greedy Unrestricted)")
        if sgru: master_summary.append(sgru)

        # Method 3: Greedy Restricted (25% Max Weight)
        # Finds the best 25 stocks but ensures no single asset exceeds 25% of the portfolio
        res_grr = mvpt.optimize_greedy_restricted(capital=CAPITAL, num_stocks=25)
        sgrr = save_result(res_grr['allocation'], "MVPT_Greedy_Restricted", tracker, "MVPT (Greedy Restricted)")
        if sgrr: master_summary.append(sgrr)

    # ---------------------------------------------------------
    # 7. FINAL SUMMARY
    # ---------------------------------------------------------
    print("\n" + "="*60)
    print("PROCESSING COMPLETE")
    print("="*60)
    
    if master_summary:
        summary_df = pd.DataFrame(master_summary)
        summary_path = os.path.join(OUTPUT_DIR, "MASTER_PERFORMANCE_SUMMARY.csv")
        summary_df.to_csv(summary_path, index=False)
        print(f"\nFinal Performance Summary (Sell Date: {SELL_DATE}) :")
        print(summary_df[['Strategy', 'Total Invested', 'Current Value', 'Total Return (%)']])
        print(f"\nAll results saved to: {os.path.abspath(OUTPUT_DIR)}")
    else:
        print("No portfolios were successfully generated.")

In [5]:
if __name__ == "__main__":
    main()

STARTING MASTER PORTFOLIO GENERATION
Training End: 2021-01-01 | Target Sell Date: 2021-09-01

[Phase 1] Fetching Core Data (CAPM Processor)...

=== Fetching Market Data ===
Initializing download for 3 unique tickers...


Initializing download for 3 unique tickers...


=== Market Data Fetch Complete ===


=== Fetching Stock Data ===
Initializing download for 78 unique tickers...


1 Failed download:
['LUNR']: YFPricesMissingError('possibly delisted; no price data found  (1d 2018-01-01 -> 2021-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1514782800, endDate = 1609477200")')
1 Failed download:
['BLD']: YFPricesMissingError('possibly delisted; no price data found  (1d 2018-01-01 -> 2021-01-01) (Yahoo error = "No data found, symbol may be delisted")')
1 Failed download:
['SPWR']: YFPricesMissingError('possibly delisted; no price data found  (1d 2018-01-01 -> 2021-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1514782800, endDate = 1609477200")')
1 Failed download:
['CFLT']: YFPricesMissingError('possibly delisted; no price data found  (1d 2018-01-01 -> 2021-01-01) (Yahoo error = "No data found, symbol may be delisted")')


Initializing download for 78 unique tickers...


...

=== Daily Stock Data Fetch Complete ===

No stocks selected. Running feature selection...

=== Analyzing Feature Importance (Window: 90 days) ===
Top 5 Low-Volatility Selections: ['CFLT', 'BLD', 'LUNR', 'META', 'SPWR']

=== Optimizing Portfolio Weights ===
Eligible stocks (Price < 250.00): 16
  [OptiTune] Selected Bounds -> n: 1.00% | m: 35.00% | Unused: $497.48

Optimization Successful.
Best Bounds -> Min %: 0.01, Max %: 0.35
Projected Unused Capital: $497.48
  Tracking performance for Algo XG (Standard) on 2021-09-01...
Fetching market data for 9 assets...
Target date set to 2021-09-01. Fetching historical close...
No stocks selected. Running feature selection...

=== Analyzing Feature Importance (Window: 90 days) ===
Top 5 Low-Volatility Selections: ['CFLT', 'BLD', 'LUNR', 'META', 'SPWR']

=== Optimizing Portfolio Weights ===
Eligible stocks (Price < 250.00): 16

Optimization Successful.
Best Bounds -> Min %: 0.06, Max %: 0.06
Projected Unused Capital: $847.30
  Tracking performance

C:\Users\USER\python\port fyp 2\MVPTs.py:13: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  self.df_change = self.price_df.pct_change().iloc[1:].fillna(0)
C:\Users\USER\python\port fyp 2\MVPTs.py:29: RuntimeWarning: invalid value encountered in cast
  num_shares = np.floor(allocated_capital / latest_prices).astype(int)
C:\Users\USER\python\port fyp 2\MVPTs.py:29: RuntimeWarning: invalid value encountered in cast
  num_shares = np.floor(allocated_capital / latest_prices).astype(int)


  Tracking performance for MVPT (Greedy Unrestricted) on 2021-09-01...
Fetching market data for 4 assets...
Target date set to 2021-09-01. Fetching historical close...


C:\Users\USER\python\port fyp 2\MVPTs.py:29: RuntimeWarning: invalid value encountered in cast
  num_shares = np.floor(allocated_capital / latest_prices).astype(int)


  Tracking performance for MVPT (Greedy Restricted) on 2021-09-01...
Fetching market data for 15 assets...
Target date set to 2021-09-01. Fetching historical close...

PROCESSING COMPLETE

Final Performance Summary (Sell Date: 2021-09-01) :
                      Strategy  Total Invested  Current Value  \
0            Algo 1 (Standard)         4625.40        5532.10   
1               Algo 1 (Naive)         4607.63        5837.12   
2            Algo 2 (Standard)         4147.30        4567.19   
3               Algo 2 (Naive)         3835.75        4181.74   
4            Algo 4 (Standard)         4240.01        7598.43   
5               Algo 4 (Naive)         3474.82        4376.39   
6           Algo XG (Standard)         4502.52        5863.60   
7              Algo XG (Naive)         4152.70        5181.40   
8           MVPT (Ledoit-Wolf)         2402.56        2453.40   
9   MVPT (Greedy Unrestricted)         1491.79        1580.79   
10    MVPT (Greedy Restricted)         2762.